# ResQCity · vision training on Google Colab

Runs the same commands as `make vision` on a Colab GPU, so the results match the local GPU run
(same code, configs and seeds). Datasets are public and need no account (see
`docs/data-sources.md`). **Runtime → Change runtime type → GPU** first.

Steps: clone repo · install · download data to Drive · annotations and video-level splits ·
train the accident window classifier · evaluate (test split) · fine-tune fire/smoke YOLO.
Outputs are written under `results/vision/` and `data/models/`; copy them back to your machine.

In [ ]:
# 1. Mount Drive so downloads/models survive the session
from google.colab import drive

drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/resqcity"

In [ ]:
# 2. Clone (replace with your repository URL) and install with uv
!git clone https://github.com/<you>/resqcity.git /content/resqcity || true
%cd /content/resqcity
!pip -q install uv
!uv sync --group vision

In [ ]:
# 3. Keep data/ on Drive (large, gitignored)
import os

os.makedirs(f"{WORK}/data", exist_ok=True)
if not os.path.islink("data"):
    !rm -rf data && ln -s {WORK}/data data

In [ ]:
# 4. Data: downloads (cap in configs/vision.yaml), annotations + video-level splits
!uv run python -m resqcity.vision.datasets.download
!uv run python -m resqcity.vision.datasets.annotations

In [ ]:
# 5. Accident window classifier (frozen MobileNetV3 features + MLP, MIL for weak labels)
!uv run python -m resqcity.vision.train_clip --run-id clip_main

In [ ]:
# 6. Evaluate on the TEST split (YOLO+ByteTrack heuristics, stacking on val, metrics, sim_inputs.yaml)
!uv run python -m resqcity.vision.eval.evaluate --run-id clip_main
!cat results/vision/clip_main/test_video_metrics.csv

In [ ]:
# 7. Fire/smoke: D-Fire -> YOLO format, fine-tune, test metrics
!uv run python -m resqcity.vision.datasets.prepare_dfire
!uv run python -m resqcity.vision.train_fire --run-id fire_main
!cat results/vision/fire_main/fire_metrics.json

In [ ]:
# 8. Copy results back to Drive (download them to your machine afterwards)
!mkdir -p {WORK}/results && cp -r results/vision {WORK}/results/